# AMIA - MHEDAS - Challenge: Part 3

#### Marc Albesa, Maria Fite, Jaume Juan

## Challenge 3 - Advanced detection algorithms

This notebook implements the missing Challenge 3 work from the project PDFs:

- Train advanced object detection models on all abnormality classes.
- Compare them against the previous baseline.
- Report overall and per-class stratified metrics.
- Select the best model.
- Generate a Kaggle-compatible `submission.csv`.

Part 2 prepared the baseline pipelines. The main correction here is to remove the single-class filter and apply the object detection workflow to the full set of abnormality classes.

## Requirements from the PDFs

The project statement describes the final task as a chest X-ray abnormality detection challenge. The relevant Challenge 3 requirements are:

- Train an advanced detection algorithm.
- Compare the results with our own baseline.
- Stratify results if useful.
- Submit predictions to Kaggle.

The challenge score is based on mean Average Precision with an IoU threshold around `0.4`. For that reason this notebook computes `AP@0.4` overall and per class, instead of relying only on the default `mAP50`/`mAP50-95` summaries.

## What was already done before Part 3

### Part 1

- Dataset loading.
- Exploratory Data Analysis.
- Class distribution, image size analysis and bounding-box visualization.

### Part 2

- Weighted Box Fusion to merge annotations from several radiologists.
- ResNet-18 classifier for normal vs abnormal X-rays.
- Single-class object detection baseline.
- YOLO, RT-DETR and Faster R-CNN pipelines.
- First validation metrics and qualitative examples.

### What changes in Part 3

The Part 2 detection code filtered one class. Challenge 3 needs the same idea, but for all classes. This notebook therefore rebuilds the detection datasets without the single-class filter.

In [ ]:
from pathlib import Path
import os
import shutil
import random
import time
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def env_flag(name, default):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

def env_int(name, default):
    value = os.environ.get(name)
    return default if value is None or value == "" else int(value)

def env_float(name, default):
    value = os.environ.get(name)
    return default if value is None or value == "" else float(value)

try:
    import torch
    TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ULTRALYTICS_DEVICE = 0 if torch.cuda.is_available() else "cpu"
except Exception:
    torch = None
    TORCH_DEVICE = "cpu"
    ULTRALYTICS_DEVICE = "cpu"

# Main paths. Override these from the VHIO launcher with AMIA_PROJECT_DIR and AMIA_BASE_DIR.
PIC_PROJECT_DIR = Path("/data/mhedas/common/jjuan/AMIA")
default_project_dir = PIC_PROJECT_DIR if PIC_PROJECT_DIR.exists() else Path.cwd()
PROJECT_DIR = Path(os.environ.get("AMIA_PROJECT_DIR", str(default_project_dir))).expanduser()
BASE_DIR = Path(os.environ.get("AMIA_BASE_DIR", "/data/mhedas/common/challenge_dataset")).expanduser()

TRAIN_DIR = BASE_DIR / "train" / "train"
TEST_DIR = BASE_DIR / "test" / "test"
WORKSPACE_DIR = PROJECT_DIR / "yolo_baseline"
YOLO_ROOT = WORKSPACE_DIR / "yolo_dataset_all_classes"

PNG_SIZE = env_int("AMIA_PNG_SIZE", 1024)
RANDOM_STATE = env_int("AMIA_RANDOM_STATE", 42)

# Grid search controls.
RUN_TRAINING = env_flag("AMIA_RUN_TRAINING", True)
RUN_GRID_SEARCH = env_flag("AMIA_RUN_GRID_SEARCH", True)
SKIP_FINISHED_RUNS = env_flag("AMIA_SKIP_FINISHED_RUNS", True)

# The grids are sized for an overnight VHIO 5090 run. Override from the launcher if needed.
YOLO_GRID_EPOCHS = env_int("AMIA_YOLO_GRID_EPOCHS", 25)
RTDETR_GRID_EPOCHS = env_int("AMIA_RTDETR_GRID_EPOCHS", 20)
FASTER_RCNN_GRID_EPOCHS = env_int("AMIA_FASTER_RCNN_GRID_EPOCHS", 7)

# Evaluation/submission configuration.
EVAL_IOU_THRESHOLD = env_float("AMIA_EVAL_IOU_THRESHOLD", 0.4)
PRED_CONF_FOR_AP = env_float("AMIA_PRED_CONF_FOR_AP", 0.001)
SUBMISSION_CONF = env_float("AMIA_SUBMISSION_CONF", 0.25)

print("Project dir:", PROJECT_DIR)
print("Dataset dir:", BASE_DIR)
print("Torch device:", TORCH_DEVICE)
print("Ultralytics device:", ULTRALYTICS_DEVICE)
print("RUN_TRAINING:", RUN_TRAINING)
print("RUN_GRID_SEARCH:", RUN_GRID_SEARCH)
print("Epochs:", YOLO_GRID_EPOCHS, RTDETR_GRID_EPOCHS, FASTER_RCNN_GRID_EPOCHS)

## Load challenge data

In [ ]:
required_files = [
    BASE_DIR / "train.csv",
    BASE_DIR / "test.csv",
    BASE_DIR / "img_size.csv",
    BASE_DIR / "sample_submission.csv",
]
missing = [str(p) for p in required_files if not p.exists()]
if missing:
    raise FileNotFoundError("Run this notebook on PIC or mount the dataset. Missing: " + ", ".join(missing))

train_df_raw = pd.read_csv(BASE_DIR / "train.csv")
test_df = pd.read_csv(BASE_DIR / "test.csv")
img_size = pd.read_csv(BASE_DIR / "img_size.csv")
sample_submission = pd.read_csv(BASE_DIR / "sample_submission.csv")

# Class 14 is the normal/no-finding label. Detection models are trained on classes 0-13.
NO_FINDING_CLASS_ID = 14
class_lookup = (
    train_df_raw[["class_id", "class_name"]]
    .drop_duplicates()
    .sort_values("class_id")
    .set_index("class_id")["class_name"]
    .to_dict()
)
DETECTION_CLASS_IDS = [cid for cid in sorted(class_lookup) if cid != NO_FINDING_CLASS_ID]
DETECTION_CLASS_NAMES = {cid: class_lookup[cid] for cid in DETECTION_CLASS_IDS}

print("Train rows:", len(train_df_raw))
print("Train images:", train_df_raw["image_id"].nunique())
print("Test images:", test_df["image_id"].nunique() if "image_id" in test_df.columns else len(test_df))
print("Detection classes:")
for cid, name in DETECTION_CLASS_NAMES.items():
    print(f"  {cid}: {name}")

display(train_df_raw.head())

## 1. Weighted Box Fusion for all classes

The original annotations can include several radiologists per image. Weighted Box Fusion reduces annotation noise by merging overlapping boxes of the same class into a consensus bounding box.

This is applied per image and per class. Normal images are kept as empty-label/background examples for detector training.

In [ ]:
try:
    from ensemble_boxes import weighted_boxes_fusion
except ImportError as exc:
    raise ImportError("Install first: pip install ensemble-boxes") from exc

def fuse_train_data_all_classes(df, img_size_df, iou_thr=0.5, skip_box_thr=0.0):
    size_df = img_size_df[["image_id", "dim0", "dim1"]].drop_duplicates("image_id")
    work = df.merge(size_df, on="image_id", how="left")
    fused_rows = []

    for img_id, group in tqdm(work.groupby("image_id"), desc="WBF per image"):
        real = group[(group["class_id"] != NO_FINDING_CLASS_ID) & group["x_min"].notna()].copy()

        if real.empty:
            fused_rows.append({
                "image_id": img_id,
                "class_id": NO_FINDING_CLASS_ID,
                "x_min": np.nan,
                "y_min": np.nan,
                "x_max": np.nan,
                "y_max": np.nan,
            })
            continue

        h = float(real["dim0"].iloc[0])
        w = float(real["dim1"].iloc[0])

        for class_id, c_group in real.groupby("class_id"):
            boxes, scores, labels = [], [], []
            for _, row in c_group.iterrows():
                x1 = np.clip(row.x_min / w, 0, 1)
                y1 = np.clip(row.y_min / h, 0, 1)
                x2 = np.clip(row.x_max / w, 0, 1)
                y2 = np.clip(row.y_max / h, 0, 1)
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                scores.append(1.0)
                labels.append(int(class_id))

            if not boxes:
                continue

            fused_boxes, fused_scores, fused_labels = weighted_boxes_fusion(
                [boxes], [scores], [labels],
                iou_thr=iou_thr,
                skip_box_thr=skip_box_thr,
            )

            for box, label in zip(fused_boxes, fused_labels):
                fused_rows.append({
                    "image_id": img_id,
                    "class_id": int(label),
                    "x_min": float(box[0] * w),
                    "y_min": float(box[1] * h),
                    "x_max": float(box[2] * w),
                    "y_max": float(box[3] * h),
                })

    fused_df = pd.DataFrame(fused_rows)
    fused_df["class_name"] = fused_df["class_id"].map(class_lookup)
    return fused_df

train_df_fused = fuse_train_data_all_classes(train_df_raw, img_size, iou_thr=0.5)

print("Fused rows:", len(train_df_fused))
print("Fused images:", train_df_fused["image_id"].nunique())
display(train_df_fused["class_name"].value_counts().rename_axis("class").reset_index(name="count"))

## 2. Model-specific input adaptation

The raw dataset is `image + CSV annotations`, but each architecture expects a different input format.

| Model | Image input | Label input | Adaptation |
|---|---|---|---|
| ResNet-18 | 224 x 224 image tensor | Normal/abnormal image label | Already done in Part 2 as baseline |
| YOLO | Image files in train/val folders | One `.txt` per image with normalized boxes | Rebuilt here for all classes |
| RT-DETR | Same Ultralytics dataset as YOLO | Same YOLO labels/YAML | Reuses the all-class YOLO dataset |
| Faster R-CNN | Image tensor | PyTorch `target` dictionary with boxes/labels | Rebuilt here for all classes |

The key Challenge 3 change is removing the single-class filter from Part 2.

## 3. Build an all-class YOLO/RT-DETR dataset

This replaces the Part 2 line that filtered a single class. Here every abnormality class `0-13` is kept. Images labeled as `No finding` are included with empty label files so the detector also sees background images.

In [ ]:
from sklearn.model_selection import train_test_split

def primary_class_for_split(group):
    real = group[group["class_id"] != NO_FINDING_CLASS_ID]
    if real.empty:
        return NO_FINDING_CLASS_ID
    return int(real["class_id"].mode().iloc[0])

primary_df = (
    train_df_raw.groupby("image_id")
    .apply(primary_class_for_split)
    .rename("primary_class")
    .reset_index()
)

stratify_values = primary_df["primary_class"]
if stratify_values.value_counts().min() < 2:
    stratify_values = None

train_ids, val_ids = train_test_split(
    primary_df["image_id"].tolist(),
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=stratify_values,
)
train_ids = set(train_ids)
val_ids = set(val_ids)

print("Train images:", len(train_ids))
print("Val images:", len(val_ids))
display(primary_df["primary_class"].map(class_lookup).value_counts().rename_axis("primary_class").reset_index(name="images"))

In [ ]:
def safe_link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    if dst.exists() or dst.is_symlink():
        return
    try:
        os.symlink(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def write_yolo_dataset_all_classes(fused_df, img_size_df, train_ids, val_ids, yolo_root):
    yolo_root = Path(yolo_root)
    if yolo_root.exists():
        shutil.rmtree(yolo_root)

    for split in ["train", "val"]:
        (yolo_root / "images" / split).mkdir(parents=True, exist_ok=True)
        (yolo_root / "labels" / split).mkdir(parents=True, exist_ok=True)

    size_df = img_size_df[["image_id", "dim0", "dim1"]].drop_duplicates("image_id")
    det_df = fused_df[fused_df["class_id"] != NO_FINDING_CLASS_ID].merge(size_df, on="image_id", how="left")
    det_by_image = {img_id: group for img_id, group in det_df.groupby("image_id")}

    all_ids = sorted(set(train_ids) | set(val_ids))
    for img_id in tqdm(all_ids, desc="Writing YOLO labels"):
        split = "train" if img_id in train_ids else "val"
        label_path = yolo_root / "labels" / split / f"{img_id}.txt"
        image_dst = yolo_root / "images" / split / f"{img_id}.png"
        image_src = TRAIN_DIR / f"{img_id}.png"

        group = det_by_image.get(img_id)
        lines = []
        if group is not None and not group.empty:
            h = float(group["dim0"].iloc[0])
            w = float(group["dim1"].iloc[0])
            for _, row in group.iterrows():
                x1 = np.clip(row["x_min"], 0, w)
                y1 = np.clip(row["y_min"], 0, h)
                x2 = np.clip(row["x_max"], 0, w)
                y2 = np.clip(row["y_max"], 0, h)
                if x2 <= x1 or y2 <= y1:
                    continue
                x_center = ((x1 + x2) / 2) / w
                y_center = ((y1 + y2) / 2) / h
                width = (x2 - x1) / w
                height = (y2 - y1) / h
                class_id = int(row["class_id"])
                lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        label_path.write_text("\n".join(lines) + ("\n" if lines else ""))
        safe_link_or_copy(image_src, image_dst)

    data_config = {
        "path": str(yolo_root),
        "train": "images/train",
        "val": "images/val",
        "names": {int(cid): str(name) for cid, name in DETECTION_CLASS_NAMES.items()},
    }
    WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)
    yaml_path = WORKSPACE_DIR / "vinbigdata_all_classes.yaml"
    with open(yaml_path, "w") as f:
        yaml.safe_dump(data_config, f, sort_keys=True)

    split_df = pd.DataFrame({
        "image_id": sorted(set(train_ids) | set(val_ids)),
        "split": ["train" if img_id in train_ids else "val" for img_id in sorted(set(train_ids) | set(val_ids))],
    })
    split_path = WORKSPACE_DIR / "all_classes_split.csv"
    split_df.to_csv(split_path, index=False)

    return yaml_path, split_path

YAML_PATH, SPLIT_PATH = write_yolo_dataset_all_classes(train_df_fused, img_size, train_ids, val_ids, YOLO_ROOT)
print("YAML:", YAML_PATH)
print("Split:", SPLIT_PATH)
print("YOLO root:", YOLO_ROOT)

In [ ]:
# Quick sanity checks: number of images and label files per split.
for split in ["train", "val"]:
    n_images = len(list((YOLO_ROOT / "images" / split).glob("*.png")))
    n_labels = len(list((YOLO_ROOT / "labels" / split).glob("*.txt")))
    print(split, "images:", n_images, "labels:", n_labels)

with open(YAML_PATH) as f:
    print(f.read())

## 4. Hyperparameter grids

Instead of training one fixed configuration, Challenge 3 now runs a bounded grid search for each detector.

The grids are intentionally not huge: all three model families should be trainable overnight on the PIC GPU. The search compares the hyperparameters that are most likely to matter in this project:

- Image size: localization resolution vs training time/memory.
- Learning rate: convergence stability.
- Optimizer and weight decay: regularization/generalization.
- Batch size: GPU memory trade-off.

For YOLO and RT-DETR, candidates are ranked with the Ultralytics validation metrics after training. For Faster R-CNN, candidates are ranked with `AP@0.4`, which matches the challenge metric more closely.

In [ ]:
YOLO_GRID = [
    {"name": "yolo_sgd_640_lr1e2", "imgsz": 640, "batch": 16, "optimizer": "SGD", "lr0": 1e-2, "weight_decay": 5e-4, "epochs": YOLO_GRID_EPOCHS},
    {"name": "yolo_sgd_800_lr5e3", "imgsz": 800, "batch": 8,  "optimizer": "SGD", "lr0": 5e-3, "weight_decay": 5e-4, "epochs": YOLO_GRID_EPOCHS},
    {"name": "yolo_adamw_640_lr1e3", "imgsz": 640, "batch": 16, "optimizer": "AdamW", "lr0": 1e-3, "weight_decay": 1e-2, "epochs": YOLO_GRID_EPOCHS},
    {"name": "yolo_sgd_512_lr1e2", "imgsz": 512, "batch": 24, "optimizer": "SGD", "lr0": 1e-2, "weight_decay": 5e-4, "epochs": YOLO_GRID_EPOCHS},
]

RTDETR_GRID = [
    {"name": "rtdetr_adamw_640_lr1e4", "imgsz": 640, "batch": 8,  "optimizer": "AdamW", "lr0": 1e-4, "weight_decay": 1e-4, "epochs": RTDETR_GRID_EPOCHS},
    {"name": "rtdetr_adamw_640_lr3e4", "imgsz": 640, "batch": 8,  "optimizer": "AdamW", "lr0": 3e-4, "weight_decay": 5e-4, "epochs": RTDETR_GRID_EPOCHS},
    {"name": "rtdetr_adamw_512_lr1e4", "imgsz": 512, "batch": 10, "optimizer": "AdamW", "lr0": 1e-4, "weight_decay": 1e-4, "epochs": RTDETR_GRID_EPOCHS},
]

FASTER_RCNN_GRID = [
    {"name": "frcnn_sgd_lr1e3", "batch": 2, "optimizer": "SGD", "lr": 1e-3, "momentum": 0.9, "weight_decay": 5e-4, "epochs": FASTER_RCNN_GRID_EPOCHS, "step_size": 3, "gamma": 0.1},
    {"name": "frcnn_sgd_lr5e4", "batch": 2, "optimizer": "SGD", "lr": 5e-4, "momentum": 0.9, "weight_decay": 5e-4, "epochs": FASTER_RCNN_GRID_EPOCHS, "step_size": 3, "gamma": 0.1},
    {"name": "frcnn_adamw_lr1e4", "batch": 2, "optimizer": "AdamW", "lr": 1e-4, "momentum": 0.0, "weight_decay": 1e-4, "epochs": FASTER_RCNN_GRID_EPOCHS, "step_size": 3, "gamma": 0.5},
]

print(f"YOLO candidates: {len(YOLO_GRID)}")
print(f"RT-DETR candidates: {len(RTDETR_GRID)}")
print(f"Faster R-CNN candidates: {len(FASTER_RCNN_GRID)}")

## 5. YOLOv8s hyperparameter grid search

YOLO is treated as the practical detection baseline. We test several image sizes, optimizers and learning rates. The best candidate is selected using validation `mAP50`, which is the closest built-in Ultralytics metric to the challenge IoU threshold.

In [ ]:
try:
    from ultralytics import YOLO
except ImportError as exc:
    raise ImportError("Install first: pip install ultralytics") from exc

def run_ultralytics_grid(model_class, base_weights, grid, data_yaml, project_dir, model_family):
    rows = []
    project_dir = Path(project_dir)

    for cfg in grid:
        run_name = cfg["name"]
        run_dir = project_dir / run_name
        best_weights = run_dir / "weights" / "best.pt"
        last_weights = run_dir / "weights" / "last.pt"

        print(f"\n=== {model_family}: {run_name} ===")
        if RUN_TRAINING and RUN_GRID_SEARCH and not (SKIP_FINISHED_RUNS and best_weights.exists()):
            model = model_class(base_weights)
            model.train(
                data=str(data_yaml),
                epochs=cfg["epochs"],
                imgsz=cfg["imgsz"],
                batch=cfg["batch"],
                device=ULTRALYTICS_DEVICE,
                project=str(project_dir),
                name=run_name,
                exist_ok=True,
                optimizer=cfg["optimizer"],
                lr0=cfg["lr0"],
                weight_decay=cfg["weight_decay"],
                patience=30,
                plots=True,
            )
        elif best_weights.exists():
            print("Skipping training; found existing weights:", best_weights)
        else:
            print("Training disabled and no existing weights found; skipping candidate.")
            continue

        weights_to_eval = best_weights if best_weights.exists() else last_weights
        if not weights_to_eval.exists():
            print("No weights found after training; skipping candidate.")
            continue

        trained_model = model_class(str(weights_to_eval))
        metrics = trained_model.val(
            data=str(data_yaml),
            split="val",
            imgsz=cfg["imgsz"],
            batch=cfg["batch"],
            device=ULTRALYTICS_DEVICE,
            verbose=False,
        )

        row = {
            "model_family": model_family,
            "run_name": run_name,
            "weights": str(weights_to_eval),
            "precision": float(metrics.box.mp),
            "recall": float(metrics.box.mr),
            "mAP50": float(metrics.box.map50),
            "mAP50-95": float(metrics.box.map),
            "fitness": float(metrics.fitness),
            **cfg,
        }
        rows.append(row)
        pd.DataFrame(rows).to_csv(project_dir / f"{model_family.lower().replace(' ', '_')}_grid_results.csv", index=False)
        print(pd.Series(row)[["run_name", "precision", "recall", "mAP50", "mAP50-95", "fitness"]])

    if not rows:
        return pd.DataFrame(), None, None

    results_df = pd.DataFrame(rows).sort_values("mAP50", ascending=False).reset_index(drop=True)
    best_row = results_df.iloc[0]
    best_model = model_class(best_row["weights"])
    return results_df, best_row, best_model

yolo_grid_results, best_yolo_row, model_yolo = run_ultralytics_grid(
    YOLO, "yolov8s.pt", YOLO_GRID, YAML_PATH, WORKSPACE_DIR, "YOLOv8s"
)

if best_yolo_row is not None:
    yolo_weights = Path(best_yolo_row["weights"])
    print("Best YOLO weights:", yolo_weights)
    display(yolo_grid_results)
else:
    yolo_weights = WORKSPACE_DIR / "yolo_all_classes" / "weights" / "best.pt"
    model_yolo = YOLO(str(yolo_weights)) if yolo_weights.exists() else None

## 6. RT-DETR hyperparameter grid search

RT-DETR is the transformer-based detector. Because it is heavier than YOLO, the grid is smaller: three configurations that vary learning rate, regularization and input resolution.

In [ ]:
try:
    from ultralytics import RTDETR
except ImportError as exc:
    raise ImportError("Install first: pip install ultralytics") from exc

rtdetr_grid_results, best_rtdetr_row, model_rtdetr = run_ultralytics_grid(
    RTDETR, "rtdetr-l.pt", RTDETR_GRID, YAML_PATH, WORKSPACE_DIR, "RT-DETR-l"
)

if best_rtdetr_row is not None:
    rtdetr_weights = Path(best_rtdetr_row["weights"])
    print("Best RT-DETR weights:", rtdetr_weights)
    display(rtdetr_grid_results)
else:
    rtdetr_weights = WORKSPACE_DIR / "rtdetr_all_classes" / "weights" / "best.pt"
    model_rtdetr = RTDETR(str(rtdetr_weights)) if rtdetr_weights.exists() else None

## 7. Faster R-CNN all-class dataset

Faster R-CNN needs a PyTorch dataset returning `image, target`, where `target` contains absolute pixel boxes and labels.

TorchVision reserves label `0` for background, so the medical classes `0-13` are shifted to `1-14` during Faster R-CNN training.

In [ ]:
import cv2
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T

class VinBigDetectionDatasetAllClasses(Dataset):
    def __init__(self, image_ids, fused_df, img_size_df, img_dir, transforms=None, png_size=1024):
        self.image_ids = list(image_ids)
        self.img_dir = Path(img_dir)
        self.transforms = transforms
        self.png_size = png_size
        self.size_df = img_size_df[["image_id", "dim0", "dim1"]].drop_duplicates("image_id").set_index("image_id")
        det_df = fused_df[fused_df["class_id"] != NO_FINDING_CLASS_ID].copy()
        self.groups = {img_id: group for img_id, group in det_df.groupby("image_id")}

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img_path = self.img_dir / f"{img_id}.png"
        img = cv2.imread(str(img_path))
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        size = self.size_df.loc[img_id]
        orig_h = float(size["dim0"])
        orig_w = float(size["dim1"])
        sx = self.png_size / orig_w
        sy = self.png_size / orig_h

        boxes = []
        labels = []
        group = self.groups.get(img_id)
        if group is not None:
            for _, row in group.iterrows():
                x1 = np.clip(row["x_min"] * sx, 0, self.png_size)
                y1 = np.clip(row["y_min"] * sy, 0, self.png_size)
                x2 = np.clip(row["x_max"] * sx, 0, self.png_size)
                y2 = np.clip(row["y_max"] * sy, 0, self.png_size)
                if x2 > x1 and y2 > y1:
                    boxes.append([x1, y1, x2, y2])
                    labels.append(int(row["class_id"]) + 1)

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0]) if len(boxes) else torch.zeros((0,), dtype=torch.float32)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(labels),), dtype=torch.int64),
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

def collate_fn(batch):
    return tuple(zip(*batch))

frcnn_transforms = T.Compose([T.ToTensor()])
train_dataset_frcnn = VinBigDetectionDatasetAllClasses(sorted(train_ids), train_df_fused, img_size, TRAIN_DIR, frcnn_transforms, PNG_SIZE)
val_dataset_frcnn = VinBigDetectionDatasetAllClasses(sorted(val_ids), train_df_fused, img_size, TRAIN_DIR, frcnn_transforms, PNG_SIZE)

train_loader_frcnn = DataLoader(train_dataset_frcnn, batch_size=FASTER_RCNN_BATCH, shuffle=True, collate_fn=collate_fn)
val_loader_frcnn = DataLoader(val_dataset_frcnn, batch_size=FASTER_RCNN_BATCH, shuffle=False, collate_fn=collate_fn)

print("Faster R-CNN train images:", len(train_dataset_frcnn))
print("Faster R-CNN val images:", len(val_dataset_frcnn))

## 8. Faster R-CNN hyperparameter grid search

Faster R-CNN is trained with PyTorch/TorchVision, so the search loop is manual. The grid varies optimizer and learning rate, then ranks candidates with `AP@0.4` on the validation split.

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def get_faster_rcnn_model(num_classes):
    weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
    model = fasterrcnn_resnet50_fpn(weights=weights)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

def make_frcnn_optimizer(model, cfg):
    params = [p for p in model.parameters() if p.requires_grad]
    if cfg["optimizer"] == "SGD":
        return torch.optim.SGD(params, lr=cfg["lr"], momentum=cfg["momentum"], weight_decay=cfg["weight_decay"])
    if cfg["optimizer"] == "AdamW":
        return torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    raise ValueError(f"Unsupported optimizer: {cfg['optimizer']}")

def evaluate_frcnn_ap04_for_grid(model, data_loader):
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[EVAL_IOU_THRESHOLD],
        class_metrics=False,
    )
    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="FRCNN grid eval", leave=False):
            images = [img.to(TORCH_DEVICE) for img in images]
            outputs = model(images)
            preds = [
                {
                    "boxes": out["boxes"].detach().cpu().float(),
                    "scores": out["scores"].detach().cpu().float(),
                    "labels": out["labels"].detach().cpu().long(),
                }
                for out in outputs
            ]
            tgts = [
                {
                    "boxes": tgt["boxes"].detach().cpu().float(),
                    "labels": tgt["labels"].detach().cpu().long(),
                }
                for tgt in targets
            ]
            metric.update(preds, tgts)
    result = metric.compute()
    return float(result["map"]), float(result["mar_100"])

def run_faster_rcnn_grid(grid):
    rows = []
    grid_dir = WORKSPACE_DIR / "faster_rcnn_grid"
    grid_dir.mkdir(parents=True, exist_ok=True)
    num_classes = len(DETECTION_CLASS_IDS) + 1

    for cfg in grid:
        run_name = cfg["name"]
        weights_path = grid_dir / f"{run_name}.pt"
        print(f"\n=== Faster R-CNN: {run_name} ===")

        model = get_faster_rcnn_model(num_classes).to(TORCH_DEVICE)

        if RUN_TRAINING and RUN_GRID_SEARCH and not (SKIP_FINISHED_RUNS and weights_path.exists()):
            train_loader_cfg = DataLoader(
                train_dataset_frcnn, batch_size=cfg["batch"], shuffle=True, collate_fn=collate_fn
            )
            optimizer = make_frcnn_optimizer(model, cfg)
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=cfg["step_size"], gamma=cfg["gamma"])

            for epoch in range(cfg["epochs"]):
                model.train()
                epoch_loss = 0.0
                start = time.time()

                for step, (images, targets) in enumerate(train_loader_cfg, start=1):
                    images = [img.to(TORCH_DEVICE) for img in images]
                    targets = [{k: v.to(TORCH_DEVICE) for k, v in t.items()} for t in targets]

                    loss_dict = model(images, targets)
                    losses = sum(loss for loss in loss_dict.values())

                    optimizer.zero_grad()
                    losses.backward()
                    optimizer.step()

                    epoch_loss += losses.item()
                    if step % 100 == 0:
                        print(f"Epoch {epoch+1}/{cfg['epochs']} | Step {step}/{len(train_loader_cfg)} | Loss {losses.item():.4f}")

                scheduler.step()
                print(f"Epoch {epoch+1} | Avg loss {epoch_loss / len(train_loader_cfg):.4f} | Time {time.time() - start:.1f}s")

            torch.save(model.state_dict(), weights_path)
            print("Saved:", weights_path)
        elif weights_path.exists():
            print("Skipping training; found existing weights:", weights_path)
            model.load_state_dict(torch.load(weights_path, map_location=TORCH_DEVICE))
        else:
            print("Training disabled and no existing weights found; skipping candidate.")
            continue

        if weights_path.exists() and not RUN_TRAINING:
            model.load_state_dict(torch.load(weights_path, map_location=TORCH_DEVICE))

        val_loader_cfg = DataLoader(
            val_dataset_frcnn, batch_size=cfg["batch"], shuffle=False, collate_fn=collate_fn
        )
        ap04, ar04 = evaluate_frcnn_ap04_for_grid(model, val_loader_cfg)

        row = {
            "model_family": "Faster R-CNN",
            "run_name": run_name,
            "weights": str(weights_path),
            "AP@0.4": ap04,
            "AR@0.4": ar04,
            **cfg,
        }
        rows.append(row)
        pd.DataFrame(rows).to_csv(grid_dir / "faster_rcnn_grid_results.csv", index=False)
        print(pd.Series(row)[["run_name", "AP@0.4", "AR@0.4"]])

    if not rows:
        return pd.DataFrame(), None, None

    results_df = pd.DataFrame(rows).sort_values("AP@0.4", ascending=False).reset_index(drop=True)
    best_row = results_df.iloc[0]
    best_model = get_faster_rcnn_model(num_classes).to(TORCH_DEVICE)
    best_model.load_state_dict(torch.load(best_row["weights"], map_location=TORCH_DEVICE))
    return results_df, best_row, best_model

faster_rcnn_grid_results, best_frcnn_row, model_frcnn = run_faster_rcnn_grid(FASTER_RCNN_GRID)

if best_frcnn_row is not None:
    faster_rcnn_weights = Path(best_frcnn_row["weights"])
    print("Best Faster R-CNN weights:", faster_rcnn_weights)
    display(faster_rcnn_grid_results)
else:
    faster_rcnn_weights = WORKSPACE_DIR / "faster_rcnn_vinbig_all_classes.pt"
    model_frcnn = None
    if faster_rcnn_weights.exists():
        model_frcnn = get_faster_rcnn_model(len(DETECTION_CLASS_IDS) + 1).to(TORCH_DEVICE)
        model_frcnn.load_state_dict(torch.load(faster_rcnn_weights, map_location=TORCH_DEVICE))

## 9. Challenge metric: AP@0.4 overall and per class

Ultralytics reports `mAP50` and `mAP50-95`, but the challenge statement mentions mAP at IoU `> 0.4`.

The next cells compute a common metric for all detectors using TorchMetrics with `iou_thresholds=[0.4]` and `class_metrics=True`. This gives:

- Overall `AP@0.4`.
- Overall `AR@0.4`.
- Per-class `AP@0.4`, which is the stratified result Maria suggested.

In [ ]:
try:
    from torchmetrics.detection.mean_ap import MeanAveragePrecision
except ImportError as exc:
    raise ImportError("Install first: pip install torchmetrics faster-coco-eval") from exc

size_lookup = img_size[["image_id", "dim0", "dim1"]].drop_duplicates("image_id").set_index("image_id")
det_fused = train_df_fused[train_df_fused["class_id"] != NO_FINDING_CLASS_ID].copy()
det_groups = {img_id: group for img_id, group in det_fused.groupby("image_id")}

def target_for_image(img_id, label_offset=0):
    size = size_lookup.loc[img_id]
    orig_h = float(size["dim0"])
    orig_w = float(size["dim1"])
    sx = PNG_SIZE / orig_w
    sy = PNG_SIZE / orig_h

    boxes = []
    labels = []
    group = det_groups.get(img_id)
    if group is not None:
        for _, row in group.iterrows():
            x1 = np.clip(row["x_min"] * sx, 0, PNG_SIZE)
            y1 = np.clip(row["y_min"] * sy, 0, PNG_SIZE)
            x2 = np.clip(row["x_max"] * sx, 0, PNG_SIZE)
            y2 = np.clip(row["y_max"] * sy, 0, PNG_SIZE)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2])
                labels.append(int(row["class_id"]) + label_offset)

    return {
        "boxes": torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4),
        "labels": torch.as_tensor(labels, dtype=torch.int64),
    }

def empty_prediction():
    return {
        "boxes": torch.zeros((0, 4), dtype=torch.float32),
        "scores": torch.zeros((0,), dtype=torch.float32),
        "labels": torch.zeros((0,), dtype=torch.int64),
    }

def summarize_map_result(model_name, result, label_offset=0):
    overall = {
        "model": model_name,
        "AP@0.4": float(result["map"]),
        "AR@0.4": float(result["mar_100"]),
    }

    per_rows = []
    classes = result.get("classes")
    map_per_class = result.get("map_per_class")
    mar_per_class = result.get("mar_100_per_class")

    if classes is not None and map_per_class is not None:
        for cls_label, ap, ar in zip(classes.cpu().tolist(), map_per_class.cpu().tolist(), mar_per_class.cpu().tolist()):
            class_id = int(cls_label) - label_offset
            if class_id not in DETECTION_CLASS_NAMES:
                continue
            per_rows.append({
                "model": model_name,
                "class_id": class_id,
                "class_name": DETECTION_CLASS_NAMES[class_id],
                "AP@0.4": float(ap),
                "AR@0.4": float(ar),
            })

    return overall, pd.DataFrame(per_rows)

In [ ]:
def evaluate_ultralytics_detector(model, model_name, image_ids, conf=PRED_CONF_FOR_AP):
    if model is None:
        print(f"Skipping {model_name}: model is not available.")
        return None

    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[EVAL_IOU_THRESHOLD],
        class_metrics=True,
    )

    for img_id in tqdm(sorted(image_ids), desc=f"Evaluating {model_name}"):
        img_path = TRAIN_DIR / f"{img_id}.png"
        result = model.predict(str(img_path), conf=conf, verbose=False)[0]

        if result.boxes is None or len(result.boxes) == 0:
            pred = empty_prediction()
        else:
            pred = {
                "boxes": result.boxes.xyxy.detach().cpu().float(),
                "scores": result.boxes.conf.detach().cpu().float(),
                "labels": result.boxes.cls.detach().cpu().long(),
            }

        target = target_for_image(img_id, label_offset=0)
        metric.update([pred], [target])

    return metric.compute()

def evaluate_faster_rcnn_detector(model, data_loader):
    metric = MeanAveragePrecision(
        box_format="xyxy",
        iou_type="bbox",
        iou_thresholds=[EVAL_IOU_THRESHOLD],
        class_metrics=True,
    )

    model.eval()
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="Evaluating Faster R-CNN"):
            images = [img.to(TORCH_DEVICE) for img in images]
            outputs = model(images)

            preds_cpu = []
            targets_cpu = []
            for output, target in zip(outputs, targets):
                preds_cpu.append({
                    "boxes": output["boxes"].detach().cpu().float(),
                    "scores": output["scores"].detach().cpu().float(),
                    "labels": output["labels"].detach().cpu().long(),
                })
                targets_cpu.append({
                    "boxes": target["boxes"].detach().cpu().float(),
                    "labels": target["labels"].detach().cpu().long(),
                })

            metric.update(preds_cpu, targets_cpu)

    return metric.compute()

## 10. Evaluate all detectors

In [ ]:
overall_rows = []
per_class_tables = []
metric_results = {}

# Load trained weights if the model variables were not created in this kernel.
if "model_yolo" not in globals() and yolo_weights.exists():
    model_yolo = YOLO(str(yolo_weights))
if "model_rtdetr" not in globals() and rtdetr_weights.exists():
    model_rtdetr = RTDETR(str(rtdetr_weights))

yolo_result = evaluate_ultralytics_detector(globals().get("model_yolo"), "YOLOv8s", val_ids)
if yolo_result is not None:
    metric_results["YOLOv8s"] = yolo_result
    overall, per_class = summarize_map_result("YOLOv8s", yolo_result, label_offset=0)
    overall_rows.append(overall)
    per_class_tables.append(per_class)

rtdetr_result = evaluate_ultralytics_detector(globals().get("model_rtdetr"), "RT-DETR-l", val_ids)
if rtdetr_result is not None:
    metric_results["RT-DETR-l"] = rtdetr_result
    overall, per_class = summarize_map_result("RT-DETR-l", rtdetr_result, label_offset=0)
    overall_rows.append(overall)
    per_class_tables.append(per_class)

if model_frcnn is not None:
    frcnn_result = evaluate_faster_rcnn_detector(model_frcnn, val_loader_frcnn)
    metric_results["Faster R-CNN"] = frcnn_result
    overall, per_class = summarize_map_result("Faster R-CNN", frcnn_result, label_offset=1)
    overall_rows.append(overall)
    per_class_tables.append(per_class)
else:
    print("Skipping Faster R-CNN evaluation: model is not available.")

if not overall_rows:
    raise RuntimeError("No detector metrics were computed. Train or load at least one detector before this cell.")

overall_df = pd.DataFrame(overall_rows).sort_values("AP@0.4", ascending=False).reset_index(drop=True)
per_class_df = pd.concat(per_class_tables, ignore_index=True) if per_class_tables else pd.DataFrame()

display(overall_df)

## 11. Stratified results per class

This is the `stratified by class` comparison discussed in the audio. It shows whether each detector performs better or worse for specific abnormalities.

In [ ]:
per_class_ap = (
    per_class_df
    .pivot_table(index=["class_id", "class_name"], columns="model", values="AP@0.4")
    .reset_index()
    .sort_values("class_id")
)

# Add an overall row at the bottom, matching the table Maria suggested.
overall_row = {"class_id": "overall", "class_name": "Overall"}
for _, row in overall_df.iterrows():
    overall_row[row["model"]] = row["AP@0.4"]

comparison_table = pd.concat([per_class_ap, pd.DataFrame([overall_row])], ignore_index=True)
display(comparison_table)

comparison_table.to_csv(PROJECT_DIR / "challenge3_stratified_results.csv", index=False)
overall_df.to_csv(PROJECT_DIR / "challenge3_overall_results.csv", index=False)
print("Saved:", PROJECT_DIR / "challenge3_stratified_results.csv")
print("Saved:", PROJECT_DIR / "challenge3_overall_results.csv")

In [ ]:
import seaborn as sns

plot_df = per_class_ap.set_index("class_name").drop(columns=["class_id"])
plt.figure(figsize=(10, 7))
sns.heatmap(plot_df, annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1)
plt.title("AP@0.4 by class and model")
plt.ylabel("Class")
plt.xlabel("Model")
plt.tight_layout()
plt.show()

overall_df.set_index("model")[["AP@0.4", "AR@0.4"]].plot(kind="bar", figsize=(8, 4), ylim=(0, 1), rot=0)
plt.title("Overall Challenge 3 metrics")
plt.ylabel("Score")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Select the final detector

The final detector is selected by the highest validation `AP@0.4`, because that is the closest metric to the challenge statement.

In [ ]:
best_model_name = overall_df.sort_values("AP@0.4", ascending=False).iloc[0]["model"]
print("Best model by AP@0.4:", best_model_name)

if best_model_name == "YOLOv8s":
    final_detector = globals().get("model_yolo")
elif best_model_name == "RT-DETR-l":
    final_detector = globals().get("model_rtdetr")
elif best_model_name == "Faster R-CNN":
    final_detector = model_frcnn
else:
    raise ValueError(f"Unknown model: {best_model_name}")

## 13. Qualitative examples

Plot validation images with fused ground truth boxes and predictions from the final Ultralytics detector. This is useful for the report/presentation.

In [ ]:
def plot_ground_truth_and_ultralytics_predictions(model, image_ids, n=3, conf=0.25):
    if model is None:
        print("No Ultralytics model available for plotting.")
        return

    candidate_ids = [img_id for img_id in image_ids if img_id in det_groups]
    sample_ids = random.sample(candidate_ids, min(n, len(candidate_ids)))

    for img_id in sample_ids:
        img_path = TRAIN_DIR / f"{img_id}.png"
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        result = model.predict(str(img_path), conf=conf, verbose=False)[0]

        fig, axes = plt.subplots(1, 2, figsize=(16, 8))
        axes[0].imshow(img)
        axes[0].set_title(f"Ground truth - {img_id}")
        axes[1].imshow(img)
        axes[1].set_title("Prediction")

        target = target_for_image(img_id, label_offset=0)
        for box, label in zip(target["boxes"].numpy(), target["labels"].numpy()):
            x1, y1, x2, y2 = box
            axes[0].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="lime", linewidth=2))
            axes[0].text(x1, max(0, y1 - 4), DETECTION_CLASS_NAMES[int(label)], color="lime", fontsize=8, backgroundcolor="black")

        if result.boxes is not None:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].detach().cpu().numpy()
                cls = int(box.cls.item())
                score = float(box.conf.item())
                axes[1].add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="red", linewidth=2))
                axes[1].text(x1, max(0, y1 - 4), f"{DETECTION_CLASS_NAMES.get(cls, cls)} {score:.2f}", color="red", fontsize=8, backgroundcolor="black")

        for ax in axes:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

if best_model_name in ["YOLOv8s", "RT-DETR-l"]:
    plot_ground_truth_and_ultralytics_predictions(final_detector, val_ids, n=3, conf=SUBMISSION_CONF)
else:
    print("Skipping qualitative plot because final model is not an Ultralytics model.")

## 14. Generate Kaggle submission

The submission format is one row per test image. `PredictionString` contains repeated groups:

`class_id confidence x_min y_min x_max y_max`

If no abnormality is detected, we output class `14` (`No finding`).

This section supports both prediction APIs used in the project:

- YOLO / RT-DETR through Ultralytics.
- Faster R-CNN through TorchVision / PyTorch.

In [ ]:
def test_image_ids_for_submission(test_df, sample_submission):
    # Kaggle defines the expected submission rows in sample_submission.csv.
    # Use it as the source of truth to preserve exactly the requested image IDs/order.
    if "image_id" in sample_submission.columns:
        return sample_submission["image_id"].astype(str).tolist()
    if "image_id" in test_df.columns:
        return test_df["image_id"].astype(str).tolist()
    return sorted(p.stem for p in TEST_DIR.glob("*.png"))

def get_original_size_for_test_image(img_id, fallback_shape=None):
    if img_id in size_lookup.index:
        size_row = size_lookup.loc[img_id]
        return float(size_row["dim0"]), float(size_row["dim1"])
    if fallback_shape is not None:
        return float(fallback_shape[0]), float(fallback_shape[1])
    return float(PNG_SIZE), float(PNG_SIZE)

def no_finding_prediction():
    return f"{NO_FINDING_CLASS_ID} 1.0 0 0 1 1"

def finalize_submission_rows(rows, output_path):
    submission = pd.DataFrame(rows)

    # Keep the sample submission order if available.
    if "image_id" in sample_submission.columns:
        submission = sample_submission[["image_id"]].merge(submission, on="image_id", how="left")
        submission["PredictionString"] = submission["PredictionString"].fillna(no_finding_prediction())

    submission.to_csv(output_path, index=False)
    return submission

def build_submission_ultralytics(model, output_path, conf=0.25):
    if model is None:
        raise ValueError("No Ultralytics model available for submission.")

    test_ids = test_image_ids_for_submission(test_df, sample_submission)
    rows = []

    for img_id in tqdm(test_ids, desc="Predicting test set with Ultralytics"):
        img_path = TEST_DIR / f"{img_id}.png"
        result = model.predict(str(img_path), conf=conf, verbose=False)[0]

        orig_h, orig_w = get_original_size_for_test_image(img_id, fallback_shape=result.orig_shape)

        # Ultralytics boxes are in PNG/model image pixel space. Convert back to original coordinate space.
        pred_h, pred_w = result.orig_shape
        sx = orig_w / pred_w
        sy = orig_h / pred_h

        parts = []
        if result.boxes is not None and len(result.boxes) > 0:
            for box in result.boxes:
                cls = int(box.cls.item())
                score = float(box.conf.item())
                x1, y1, x2, y2 = box.xyxy[0].detach().cpu().numpy()
                x1 = np.clip(x1 * sx, 0, orig_w)
                x2 = np.clip(x2 * sx, 0, orig_w)
                y1 = np.clip(y1 * sy, 0, orig_h)
                y2 = np.clip(y2 * sy, 0, orig_h)
                if x2 <= x1 or y2 <= y1:
                    continue
                parts.extend([str(cls), f"{score:.6f}", f"{x1:.1f}", f"{y1:.1f}", f"{x2:.1f}", f"{y2:.1f}"])

        rows.append({"image_id": img_id, "PredictionString": " ".join(parts) if parts else no_finding_prediction()})

    return finalize_submission_rows(rows, output_path)

def build_submission_faster_rcnn(model, output_path, conf=0.25):
    if model is None:
        raise ValueError("No Faster R-CNN model available for submission.")

    test_ids = test_image_ids_for_submission(test_df, sample_submission)
    rows = []
    model.eval()

    with torch.no_grad():
        for img_id in tqdm(test_ids, desc="Predicting test set with Faster R-CNN"):
            img_path = TEST_DIR / f"{img_id}.png"
            img_bgr = cv2.imread(str(img_path))
            if img_bgr is None:
                raise FileNotFoundError(f"Image not found: {img_path}")
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            pred_h, pred_w = img_rgb.shape[:2]

            image_tensor = T.ToTensor()(img_rgb).to(TORCH_DEVICE)
            output = model([image_tensor])[0]

            boxes = output["boxes"].detach().cpu().numpy()
            scores = output["scores"].detach().cpu().numpy()
            labels = output["labels"].detach().cpu().numpy()

            orig_h, orig_w = get_original_size_for_test_image(img_id, fallback_shape=(pred_h, pred_w))
            sx = orig_w / pred_w
            sy = orig_h / pred_h

            parts = []
            for box, score, label in zip(boxes, scores, labels):
                if score < conf:
                    continue

                # Faster R-CNN labels are shifted by +1 because label 0 is background.
                class_id = int(label) - 1
                if class_id not in DETECTION_CLASS_NAMES:
                    continue

                x1, y1, x2, y2 = box
                x1 = np.clip(x1 * sx, 0, orig_w)
                x2 = np.clip(x2 * sx, 0, orig_w)
                y1 = np.clip(y1 * sy, 0, orig_h)
                y2 = np.clip(y2 * sy, 0, orig_h)
                if x2 <= x1 or y2 <= y1:
                    continue

                parts.extend([str(class_id), f"{float(score):.6f}", f"{x1:.1f}", f"{y1:.1f}", f"{x2:.1f}", f"{y2:.1f}"])

            rows.append({"image_id": img_id, "PredictionString": " ".join(parts) if parts else no_finding_prediction()})

    return finalize_submission_rows(rows, output_path)

submission_path = PROJECT_DIR / "submission_challenge3.csv"

if best_model_name in ["YOLOv8s", "RT-DETR-l"]:
    submission_df = build_submission_ultralytics(final_detector, submission_path, conf=SUBMISSION_CONF)
elif best_model_name == "Faster R-CNN":
    submission_df = build_submission_faster_rcnn(final_detector, submission_path, conf=SUBMISSION_CONF)
else:
    raise ValueError(f"Unknown model type: {best_model_name}")

print("Saved:", submission_path)
display(submission_df.head())

## 15. Final interpretation for report/presentation

Use the results from this notebook to write the Challenge 3 section:

- The previous ResNet-18 model is the baseline for normal/abnormal classification.
- YOLOv8s is the practical detection baseline.
- RT-DETR is the advanced transformer detector.
- Faster R-CNN is the two-stage detector comparison.
- Each detector is now trained with a small hyperparameter grid, not one fixed setup.
- The main table should include overall `AP@0.4` plus per-class stratified `AP@0.4`.
- The final Kaggle submission should use the model with the highest validation `AP@0.4`, unless speed/simplicity is prioritized.

Presentation structure:

1. Introduction and problem definition.
2. Dataset description.
3. EDA.
4. Preprocessing: WBF and model-specific input adaptation.
5. Baseline: ResNet-18 normal vs abnormal.
6. Challenge 3 detectors and hyperparameter search.
7. Overall and stratified per-class results.
8. Final model and Kaggle submission.